# **How to Build and Fine-Tune a Small Language Model**

*A Step-by-Step Guide for Beginners, Researchers, and Non-Programmers*

1) Paper Version: https://www.amazon.com/dp/B0G3MYWTJK

2) PDF version: https://leanpub.com/howtobuildandfine-tuneasmalllanguagemodel

Reference:

Liu, J. Paul. 2025. How to Build and Fine-Tune a Small Language Model. Leanpub. 479 p.

## Chapter 4  Part 1: Understanding Tokenization

Before collecting data, understand how it will be processed. Remember from Chapter 2:

**What you'll learn:**
- Where to find training data for any domain
- How to clean and format text properly
- Tokenization strategies and when to use each
- How much data you actually need (it's less than you think)
- Building a complete data pipeline from scratch

**What you'll build:** Three production-ready datasets for different use cases, plus a reusable data processing pipeline.

## Part 1: Understanding Tokenization

Before collecting data, understand how it will be processed. Remember from Chapter 2:

### Word-Level Tokenization

In [ ]:
# Character-level tokenization (what you built)
text = "Hello world"
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]

### Character-Level Tokenization

In [ ]:
# Simple implementation
text = "Hello world"
chars = sorted(list(set(text)))  # [' ', 'H', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
vocab_size = len(chars)  # 9 characters

stoi = {ch: i for i, ch in enumerate(chars)}
encoded = [stoi[c] for c in text]  # [1, 3, 5, 5, 6, 0, 8, 6, 7, 5, 2]

### Word-Level Tokenization

In [ ]:
text = "Hello world! Hello again."
words = text.lower().split()  # ['hello', 'world!', 'hello', 'again.']

vocab = sorted(set(words))  # Problem: 'world!' vs 'world'
vocab_size = len(vocab)  # Grows indefinitely

## Part 2: Practical Tokenization with Hugging Face

Let's implement all three approaches with real code.

### Setup

In [ ]:
!pip install transformers datasets tokenizers -q

import torch
from transformers import GPT2Tokenizer, AutoTokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

### Example 1: Using Pre-trained GPT-2 Tokenizer

In [ ]:
# Load GPT-2's tokenizer (what you used in Chapter 3)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Test it
text = "Hello world! This is a test of tokenization."
tokens = tokenizer.encode(text)
print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

# Decode back
decoded = tokenizer.decode(tokens)
print(f"Decoded: {decoded}")

# See actual token strings
token_strings = [tokenizer.decode([t]) for t in tokens]
print(f"Token strings: {token_strings}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Text: Hello world! This is a test of tokenization.
Tokens: [15496, 995, 0, 770, 318, 257, 1332, 286, 11241, 1634, 13]
Number of tokens: 11
Decoded: Hello world! This is a test of tokenization.
Token strings: ['Hello', ' world', '!', ' This', ' is', ' a', ' test', ' of', ' token', 'ization', '.']


### Example 2: Training Your Own BPE Tokenizer

BPE stands for **Byte Pair Encoding**, and it’s one of the most common subword tokenization algorithms used in modern NLP models (like GPT, BERT, and Whisper).

When should you train your own? When your domain has special vocabulary that GPT-2's tokenizer doesn't handle well.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# Example: Medical text with specialized terms
medical_texts = [
    "The patient presented with acute myocardial infarction.",
    "Administer 325mg aspirin and 5mg morphine sulfate IV.",
    "ECG shows ST-segment elevation in leads V1-V4.",
    "Troponin levels elevated to 15.2 ng/mL.",
    # ... add 1000+ more medical texts
]

# Create a BPE tokenizer from scratch
tokenizer = Tokenizer(models.BPE())

# Configure preprocessing
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Train on your domain data
trainer = trainers.BpeTrainer(
    vocab_size=10000,  # Smaller vocab for specialized domain
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
)

# Train (in practice, use a file with lots of text)
tokenizer.train_from_iterator(medical_texts, trainer=trainer)

# Test
text = "Patient shows ST-segment elevation myocardial infarction"
encoding = tokenizer.encode(text)
print(f"Tokens: {encoding.tokens}")
print(f"IDs: {encoding.ids}")

Tokens: ['ati', 'ent', 'shows', 'ST', '-', 'segment', 'elevation', 'myocardial', 'infarction']
IDs: [51, 53, 133, 62, 4, 132, 116, 142, 135]



### Example 3: Character-Level (Chapter 2 Approach)


In [ ]:
# Build from Chapter 2
text = "Hello world"
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Test
encoded = encode(text)
print(f"Character tokens: {encoded}")
print(f"Length: {len(encoded)} (vs {len(text.split())} words)")

Character tokens: [1, 3, 4, 4, 5, 0, 7, 5, 6, 4, 2]
Length: 11 (vs 2 words)


## Part 3: Where to Find Training Data


You need data. Lots of it. Here are your options.

### Option 1: Public Datasets

#### Wikipedia

**What:** Free, high-quality encyclopedia articles in 300+ languages.

**Size:** ~20GB (English), 6M+ articles

**How to download:**

## 🔑 Getting Your Hugging Face Token and Saving It in Google Colab

Before using Hugging Face datasets or models, you need to log in with your personal access token.

### Step 1 – Create a Hugging Face Account
1. Go to [https://huggingface.co](https://huggingface.co) and sign up (or log in if you already have an account).  
2. In the top-right corner, click your **profile photo → Settings → Access Tokens**.  
3. Click **“New token”**, give it a name (for example, `colab`), choose the **“Read”** role, and click **Create Token**.  
4. Copy the generated token — it will look like this:

hf_A1B2C3D4E5F6G7H8I9J0…

### Step 2 – Store the Token Securely in Colab
In a Google Colab notebook:

1. Open the left-hand **“🔐 Secrets”** tab (key icon).  
2. Click **“+ Add new secret”**.  
3. Enter:
- **Name:** `HF_TOKEN`  
- **Value:** *(paste your Hugging Face token)*  

Now your token is stored securely in your Colab environment.

First, we can load a smaller dataset for demonstration purposes

In [ ]:
# Load a smaller dataset for demo
from datasets import load_dataset
print("Downloading SQuAD dataset...")
dataset = load_dataset("squad", split='train[:1000]') # Load only the first 1000 examples

print(f"Total examples: {len(dataset):,}")
print(f"\nFirst example:")
print(dataset[0]['title'])
print(dataset[0]['context'][:5000])
print(dataset[0]['question'])
print(dataset[0]['answers'])

Total examples: 1,000

First example:
University_of_Notre_Dame
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [ ]:
#take at least 20-30 minutes
from datasets import load_dataset

print("Downloading English Wikipedia (modern version)...")
dataset = load_dataset("wikimedia/wikipedia", "20231101.en")  # use a newer dump version

print(f"Total articles: {len(dataset['train']):,}")
print("\nFirst article:")
print(dataset['train'][0]['title'])
print(dataset['train'][0]['text'][:500])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

20231101.en/train-00000-of-00041.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

20231101.en/train-00001-of-00041.parquet:   0%|          | 0.00/351M [00:00<?, ?B/s]

20231101.en/train-00002-of-00041.parquet:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

20231101.en/train-00013-of-00041.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

20231101.en/train-00014-of-00041.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

20231101.en/train-00015-of-00041.parquet:   0%|          | 0.00/235M [00:00<?, ?B/s]

20231101.en/train-00016-of-00041.parquet:   0%|          | 0.00/503M [00:00<?, ?B/s]

KeyboardInterrupt: 

#### Common Crawl

**What:** Raw web scrapes. Billions of web pages.

**Size:** Petabytes (yes, really)

**Quality:** Mixed. Contains gold and garbage.

**How to use:**

In [ ]:
# Don't download raw Common Crawl!
# Use pre-filtered versions like C4 or OSCAR

from datasets import load_dataset

# English C4, streamed (no full download)
ds = load_dataset("allenai/c4", "en", streaming=True)

# Peek at a few samples
for i, ex in enumerate(ds["train"].take(5), 1):
    print(f"\n{'='*50}\nSample {i}\n")
    print(ex["text"][:300])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]


Sample 1

Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class fo

Sample 2

Discussion in 'Mac OS X Lion (10.7)' started by axboi87, Jan 20, 2012.
I've got a 500gb internal drive and a 240gb SSD.
When trying to restore using disk utility i'm given the error "Not enough space on disk ____ to restore"
But I shouldn't have to do that!!!
Any ideas or workarounds before resortin

Sample 3

Foil plaid lycra and spandex shortall with metallic slinky insets. Attached metallic elastic belt with O-ring. Headband included. Great hip hop or jazz dance costume. Made in the USA.

Sample 4

How many backlinks per day for new site?
Discussion in 'Black Hat SEO' started by Omoplata, Dec 3, 2010.
1) for a newly created site, what's the max # backlinks per day I

#### Books Corpus

**What:** 11,000 unpublished books, fiction and non-fiction.

**Size:** ~5GB

**Quality:** High (actual books)

**Status:** Restricted due to copyright concerns

**Alternative:** Use openly licensed books from Project Gutenberg.

In [ ]:
# Project Gutenberg example
!wget https://www.gutenberg.org/files/1342/1342-0.txt  # Pride and Prejudice

with open('1342-0.txt', 'r', encoding='utf-8') as f:
    book = f.read()

print(f"Length: {len(book):,} characters")
print(book[:500])

--2025-10-06 13:12:46--  https://www.gutenberg.org/files/1342/1342-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 752583 (735K) [text/plain]
Saving to: ‘1342-0.txt’

1342-0.txt          100%[===================>] 734.94K  2.35MB/s    in 0.3s    

2025-10-06 13:12:47 (2.35 MB/s) - ‘1342-0.txt’ saved [752583/752583]

Length: 728,846 characters
*** START OF THE PROJECT GUTENBERG EBOOK 1342 ***




                            [Illustration:

                             GEORGE ALLEN
                               PUBLISHER

                        156 CHARING CROSS ROAD
                                LONDON

                             RUSKIN HOUSE
                                   ]

                            [Illustration:

               _Reading Jane’s Letters._      _Chap 34._
         

### Option 2: Domain-Specific Sources

#### Academic Papers (ArXiv, PubMed)

## 🔑 Getting Your Hugging Face Token and Saving It in Google Colab

Before using Hugging Face datasets or models, you need to log in with your personal access token.

### Step 1 – Create a Hugging Face Account
1. Go to [https://huggingface.co](https://huggingface.co) and sign up (or log in if you already have an account).  
2. In the top-right corner, click your **profile photo → Settings → Access Tokens**.  
3. Click **“New token”**, give it a name (for example, `colab`), choose the **“Read”** role, and click **Create Token**.  
4. Copy the generated token — it will look like this:

hf_A1B2C3D4E5F6G7H8I9J0…

### Step 2 – Store the Token Securely in Colab
In a Google Colab notebook:

1. Open the left-hand **“🔐 Secrets”** tab (key icon).  
2. Click **“+ Add new secret”**.  
3. Enter:
- **Name:** `HF_TOKEN`  
- **Value:** *(paste your Hugging Face token)*  

Now your token is stored securely in your Colab environment.

### Step 3 – Use the Token in Code
Access it in Python like this:

```python
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])



###ArXiv (AI subset; handy for quick tests)

In [ ]:
from datasets import load_dataset

# This one actually works - ArXiv papers in parquet format
dataset = load_dataset(
    "ccdv/arxiv-summarization",
    split="train",
    streaming=True
)

arxiv_texts = []
for i, item in enumerate(dataset):
    if i >= 5000:
        break
    # Has 'article' and 'abstract' fields
    arxiv_texts.append(item['article'])

    if i % 1000 == 0:
        print(f"Processed {i} papers...")

print(f"✅ Collected {len(arxiv_texts)} ArXiv papers")

README.md: 0.00B [00:00, ?B/s]

Processed 0 papers...
Processed 1000 papers...
Processed 2000 papers...
Processed 3000 papers...
Processed 4000 papers...
✅ Collected 5000 ArXiv papers


In [ ]:
from datasets import load_dataset

# ArXiv with abstracts and summaries
dataset = load_dataset("ccdv/arxiv-summarization", split="train[:500]")

print(f"Loaded {len(dataset)} papers")
print(f"Fields: {dataset[0].keys()}")

# Extract texts
arxiv_texts = [item['article'] for item in dataset]
abstracts = [item['abstract'] for item in dataset]

Loaded 500 papers
Fields: dict_keys(['article', 'abstract'])


**PubMed**

In [ ]:
from datasets import load_dataset

pubmed = load_dataset("ccdv/pubmed-summarization", split="train")
print(pubmed)
print(pubmed[0].keys())     # ['article', 'abstract', ...]
print(pubmed[0]["article"][:500])

Dataset({
    features: ['article', 'abstract'],
    num_rows: 119924
})
dict_keys(['article', 'abstract'])
a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . 
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . 
 the prevalence of malnutrition among elementary school aged ch


In [ ]:
pubmed_qa = load_dataset("pubmed_qa", "pqa_labeled")
print(pubmed_qa["train"][0])

{'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells i


#### Legal Documents

In [ ]:
from datasets import load_dataset

# US Congressional bills
dataset = load_dataset("billsum", split="train")

print(f"✅ Loaded {len(dataset)} legislative bills")

# Extract bill texts
bills = [item['text'] for item in dataset]
summaries = [item['summary'] for item in dataset]

print(f"Sample bill: {bills[0][:300]}...")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

✅ Loaded 18949 legislative bills
Sample bill: SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT ORGANIZATIONS.

    (a) Definitions.--In this section:
            (1) Business entity.--The term ``business entity'' means a 
        firm, corporation, association, partnership, consortium, joint 
   ...


In [ ]:
from datasets import load_dataset

# Option 1: EUR-Lex case classification
dataset1 = load_dataset("lex_glue", "eurlex", split="train")

# Option 2: SCOTUS case classification
dataset2 = load_dataset("lex_glue", "scotus", split="train")

# Option 3: Legal contracts (LEDGAR)
dataset3 = load_dataset("lex_glue", "ledgar", split="train")

# Option 4: UNFAIR-ToS (Terms of Service)
dataset4 = load_dataset("lex_glue", "unfair_tos", split="train")

print(f"EUR-Lex: {len(dataset1)} docs")
print(f"SCOTUS: {len(dataset2)} docs")
print(f"LEDGAR: {len(dataset3)} docs")
print(f"Unfair ToS: {len(dataset4)} docs")

# Extract texts
eurlex_texts = [item['text'] for item in dataset1]

README.md: 0.00B [00:00, ?B/s]

eurlex/train-00000-of-00001.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

eurlex/test-00000-of-00001.parquet:   0%|          | 0.00/24.3M [00:00<?, ?B/s]

eurlex/validation-00000-of-00001.parquet:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/55000 [00:00<?, ? examples/s]

...

unfair_tos/validation-00000-of-00001.par(…):   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]

EUR-Lex: 55000 docs
SCOTUS: 5000 docs
LEDGAR: 60000 docs
Unfair ToS: 5532 docs


#### Code Repositories

In [ ]:
from datasets import load_dataset

# The Pile contains GitHub code
dataset = load_dataset(
    "monology/pile-uncopyrighted",
    split="train",
    streaming=True
)

# Filter for code (GitHub subset)
code_samples = []
for i, item in enumerate(dataset):
    if i >= 50000:  # Check more samples
        break

    # Check if it's from GitHub
    if item.get('meta', {}).get('pile_set_name') == 'Github':
        code_samples.append(item['text'])

        if len(code_samples) <= 3:
            print(f"\n{'='*50}")
            print(f"Sample {len(code_samples)}:")
            print(item['text'][:300])

        if len(code_samples) >= 1000:
            break

    if i % 5000 == 0:
        print(f"Checked {i} samples, found {len(code_samples)} code files...")

print(f"\n✅ Found {len(code_samples)} code samples")

README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Checked 0 samples, found 0 code files...

Sample 1:
<?xml version="1.0" encoding="UTF-8"?>
<segment>
    <name>PD1</name>
    <description>Patient Additional Demographic</description>
    <elements>
        <field minOccurs="0" maxOccurs="0">
            <name>PD1.1</name>
            <description>Living Dependency</description>
            <

Sample 2:
{
  "fpsLimit": 60,
  "preset": "basic",
  "background": {
    "color": "#0d47a1",
    "image": "",
    "position": "50% 50%",
    "repeat": "no-repeat",
    "size": "cover"
  }
}

Sample 3:
The two classes `KinesisRecorder` and `KinesisFirehoseRecorder` allow you to interface with Amazon Kinesis Data Streams and Amazon Kinesis Data Firehose to stream analytics data for real-time processing.

## What is Amazon Kinesis Data Streams?

[Amazon Kinesis Data Streams](http://aws.amazon.com/ki
Checked 5000 samples, found 484 code files...

✅ Found 1000 code samples



### Option 3: Your Own Data

**Best option when:**
- You have proprietary data
- You need maximum relevance
- Privacy is critical

**Sources:**
- Company documents
- Customer support logs
- Internal emails (with permission!)
- Research notes
- Product descriptions
- Your own writing

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_docs(url):
    """Scrape text from documentation pages"""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Remove navigation, headers, footers
    for element in soup(['nav', 'header', 'footer', 'script', 'style']):
        element.decompose()

    # Extract text
    text = soup.get_text(separator='\n', strip=True)
    return text

# Example usage
docs_url = "https://abc.com"
text = scrape_docs(docs_url)
print(f"Extracted {len(text):,} characters")
print(text[0:500])

Extracted 14,778 characters
ABC Network - ABC.com
Skip to Content
Shifting Gears
Watch the season premiere with no sign-in needed!
Abbott Elementary
Watch the season premiere with no sign-in needed!
The Golden Bachelor
Catch up with every episode so far including the latest all-new one
High Potential
Catch up with every episode of the season so far including the latest all-new one
Dancing with the Stars
Catch up on the season so far including the latest all-new episode
WATCH NOW
STREAM ON
---STREAM ON hulu---
ABC SHOWS, SP


## Part 4: Cleaning and Formatting Data

Raw data is messy. Here's how to clean it.

In [ ]:
import re
from typing import List

class TextCleaner:
    """Production-quality text cleaning pipeline"""

    @staticmethod
    def remove_urls(text: str) -> str:
        """Remove URLs"""
        return re.sub(r'http\S+|www.\S+', '', text)

    @staticmethod
    def remove_html(text: str) -> str:
        """Remove HTML tags"""
        return re.sub(r'<[^>]+>', '', text)

    @staticmethod
    def normalize_whitespace(text: str) -> str:
        """Normalize whitespace"""
        # Replace multiple spaces/newlines with single ones
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    @staticmethod
    def remove_short_lines(text: str, min_length: int = 50) -> str:
        """Remove lines shorter than min_length characters"""
        lines = text.split('\n')
        lines = [line for line in lines if len(line.strip()) >= min_length]
        return '\n'.join(lines)

    @staticmethod
    def remove_duplicates(texts: List[str]) -> List[str]:
        """Remove exact duplicate documents"""
        seen = set()
        unique = []
        for text in texts:
            if text not in seen:
                seen.add(text)
                unique.append(text)
        return unique

    @classmethod
    def clean(cls, text: str) -> str:
        """Apply all cleaning steps"""
        text = cls.remove_urls(text)
        text = cls.remove_html(text)
        text = cls.normalize_whitespace(text)
        return text

# Example usage
dirty_text = """
<html><body>
Check out https://example.com for more info!

This is     a test    with    extra    spaces.


And too many blank lines.
</body></html>
"""

clean_text = TextCleaner.clean(dirty_text)
print("BEFORE:")
print(dirty_text)
print("\nAFTER:")
print(clean_text)

BEFORE:

<html><body>
Check out https://example.com for more info!

This is     a test    with    extra    spaces.


And too many blank lines.
</body></html>


AFTER:
Check out for more info! This is a test with extra spaces. And too many blank lines.


### Deduplication: Critical for Quality

**Why deduplicate?**
- Prevents memorization of repeated content
- Reduces overfitting
- Makes training more efficient
- Improves generalization

**Three levels of deduplication:**

#### 1. Exact Deduplication (Easy)

In [ ]:
def exact_deduplicate(texts: List[str]) -> List[str]:
    """Remove exact duplicates"""
    return list(dict.fromkeys(texts))  # Preserves order

# Example
texts = ["Hello world", "Goodbye", "Hello world", "Hi there"]
unique = exact_deduplicate(texts)
print(f"Original: {len(texts)}, Unique: {len(unique)}")

Original: 4, Unique: 3


#### 2. Near-Duplicate Detection (Medium)

In [ ]:
from difflib import SequenceMatcher

def similarity(text1: str, text2: str) -> float:
    """Calculate similarity between two texts"""
    return SequenceMatcher(None, text1, text2).ratio()

def fuzzy_deduplicate(texts: List[str], threshold: float = 0.9) -> List[str]:
    """Remove near-duplicates"""
    unique = []
    for text in texts:
        is_duplicate = False
        for existing in unique:
            if similarity(text, existing) > threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            unique.append(text)
    return unique

# Example
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "The quick brown fox jumps over the lazy dog!",  # Very similar
    "A completely different sentence here.",
]
unique = fuzzy_deduplicate(texts, threshold=0.9)
print(f"Removed {len(texts) - len(unique)} near-duplicates")

Removed 1 near-duplicates


#### 3. MinHash LSH (Advanced, for large datasets)

In [ ]:
# For datasets > 100K documents
!pip install datasketch -q

from datasketch import MinHash, MinHashLSH

def create_minhash(text: str, num_perm: int = 128) -> MinHash:
    """Create MinHash signature"""
    mh = MinHash(num_perm=num_perm)
    for word in text.lower().split():
        mh.update(word.encode('utf8'))
    return mh

def minhash_deduplicate(texts: List[str], threshold: float = 0.9) -> List[str]:
    """Fast near-duplicate detection using MinHash LSH"""
    lsh = MinHashLSH(threshold=threshold, num_perm=128)

    # Index all documents
    signatures = {}
    for i, text in enumerate(texts):
        mh = create_minhash(text)
        signatures[i] = mh
        lsh.insert(i, mh)

    # Find duplicates
    duplicates = set()
    for i in range(len(texts)):
        results = lsh.query(signatures[i])
        if len(results) > 1:  # Found duplicates
            # Keep only the first occurrence
            duplicates.update(results[1:])

    # Return only non-duplicates
    return [text for i, text in enumerate(texts) if i not in duplicates]

# This is MUCH faster for large datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 3.9 MB/s eta 0:00:00


### Formatting for Training

Models expect data in specific formats. Here are the three most common:

#### Format 1: Plain Text (Simplest)


In [ ]:
# Just concatenate documents with separators
def format_plain_text(documents: List[str]) -> str:
    """Simple concatenation with document separator"""
    separator = "\n\n### END OF DOCUMENT ###\n\n"
    return separator.join(documents)

docs = [
    "First document text here.",
    "Second document text here.",
    "Third document text here.",
]

formatted = format_plain_text(docs)
print(formatted)

First document text here.

### END OF DOCUMENT ###

Second document text here.

### END OF DOCUMENT ###

Third document text here.


#### Format 2: JSONL (Most Common)

In [ ]:
import json

def save_as_jsonl(documents: List[str], filename: str):
    """Save as JSON Lines format (one JSON object per line)"""
    with open(filename, 'w') as f:
        for doc in documents:
            json.dump({"text": doc}, f)
            f.write('\n')

# Save
save_as_jsonl(docs, "training_data.jsonl")

# Load
def load_jsonl(filename: str) -> List[str]:
    """Load from JSONL"""
    texts = []
    with open(filename, 'r') as f:
        for line in f:
            data = json.load(line)
            texts.append(data['text'])
    return texts


#### Format 3: Tokenized Tensor (Fastest)

In [ ]:
import torch
from transformers import GPT2Tokenizer

def create_tokenized_dataset(
    texts: List[str],
    tokenizer,
    max_length: int = 512,
    output_file: str = "tokenized_data.pt"
):
    """Pre-tokenize and save as PyTorch tensor"""
    all_tokens = []

    for text in texts:
        tokens = tokenizer.encode(text, max_length=max_length, truncation=True)
        all_tokens.extend(tokens)

    # Save as tensor
    tensor = torch.tensor(all_tokens, dtype=torch.long)
    torch.save(tensor, output_file)

    print(f"Saved {len(all_tokens):,} tokens to {output_file}")
    return tensor

# Usage
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tensor = create_tokenized_dataset(docs, tokenizer)

Saved 15 tokens to tokenized_data.pt


## Part 5: Complete Data Pipeline

In [ ]:
import os
import json
from typing import List, Dict
from datasets import load_dataset
from transformers import GPT2Tokenizer
import torch

class DataPipeline:
    """Complete data preparation pipeline"""

    def __init__(self, tokenizer_name: str = "gpt2"):
        self.tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_name)
        self.cleaner = TextCleaner()

    def load_from_source(self, source: str, limit: int = None) -> List[str]:
        """Load data from various sources"""
        if source == "wikipedia":
            dataset = load_dataset("wikipedia", "20220301.en", split="train")
            texts = [item['text'] for item in dataset]
        elif source.endswith('.jsonl'):
            texts = self.load_jsonl(source)
        elif source.endswith('.txt'):
            with open(source, 'r') as f:
                text = f.read()
            texts = [text]  # Single document
        else:
            raise ValueError(f"Unknown source: {source}")

        if limit:
            texts = texts[:limit]

        return texts

    def process(
        self,
        texts: List[str],
        min_length: int = 100,
        max_length: int = 512,
        deduplicate: bool = True
    ) -> Dict:
        """Complete processing pipeline"""
        print(f"Starting with {len(texts):,} documents")

        # 1. Clean
        print("Cleaning...")
        texts = [self.cleaner.clean(text) for text in texts]

        # 2. Filter short documents
        print(f"Filtering documents shorter than {min_length} characters...")
        texts = [t for t in texts if len(t) >= min_length]
        print(f"  Remaining: {len(texts):,}")

        # 3. Deduplicate
        if deduplicate:
            print("Deduplicating...")
            original_count = len(texts)
            texts = list(dict.fromkeys(texts))
            print(f"  Removed {original_count - len(texts):,} duplicates")

        # 4. Tokenize
        print("Tokenizing...")
        all_tokens = []
        for text in texts:
            tokens = self.tokenizer.encode(
                text,
                max_length=max_length,
                truncation=True
            )
            all_tokens.extend(tokens)
            all_tokens.append(self.tokenizer.eos_token_id)  # Add separator

        print(f"Total tokens: {len(all_tokens):,}")

        # 5. Split train/val
        split_idx = int(0.9 * len(all_tokens))
        train_tokens = all_tokens[:split_idx]
        val_tokens = all_tokens[split_idx:]

        return {
            'train': torch.tensor(train_tokens, dtype=torch.long),
            'val': torch.tensor(val_tokens, dtype=torch.long),
            'num_documents': len(texts),
            'num_tokens': len(all_tokens),
            'vocab_size': len(self.tokenizer),
        }

    def save(self, data: Dict, output_dir: str):
        """Save processed data"""
        os.makedirs(output_dir, exist_ok=True)

        torch.save(data['train'], f"{output_dir}/train.pt")
        torch.save(data['val'], f"{output_dir}/val.pt")

        # Save metadata
        metadata = {
            'num_documents': data['num_documents'],
            'num_tokens': data['num_tokens'],
            'vocab_size': data['vocab_size'],
        }
        with open(f"{output_dir}/metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f"Saved to {output_dir}/")

    @staticmethod
    def load_jsonl(filename: str) -> List[str]:
        """Load JSONL file"""
        texts = []
        with open(filename, 'r') as f:
            for line in f:
                data = json.loads(line)
                texts.append(data['text'])
        return texts

# Example usage: Process Wikipedia data
pipeline = DataPipeline()

# Load sample data
texts = [
    "This is the first document about machine learning.",
    "This is the second document about deep learning and neural networks.",
    "The third document discusses natural language processing.",
    # In practice, load thousands of documents
]

# Process
data = pipeline.process(
    texts,
    min_length=20,
    max_length=512,
    deduplicate=True
)

# Save
pipeline.save(data, "./processed_data")

print("\nDataset ready for training!")
print(f"Training tokens: {len(data['train']):,}")
print(f"Validation tokens: {len(data['val']):,}")

Starting with 3 documents
Cleaning...
Filtering documents shorter than 20 characters...
  Remaining: 3
Deduplicating...
  Removed 0 duplicates
Tokenizing...
Total tokens: 32
Saved to ./processed_data/

Dataset ready for training!
Training tokens: 28
Validation tokens: 4



## Part 6: Real Examples - Three Complete Datasets

### Example 1: Medical Q&A Dataset

In [ ]:
pip install "datasets<3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
# Medical literature from PubMed 3.65G
from datasets import load_dataset

# Correct dataset name and structure
dataset = load_dataset("scientific_papers", "pubmed", split="train[:5000]")

# Extract texts (correct field name)
medical_texts = [item['article'] for item in dataset if item.get('article')]

print(f"Loaded {len(medical_texts)} abstracts")
print(f"Example: {medical_texts[0][:200]}...")

The repository for scientific_papers contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/scientific_papers.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

Loaded 4866 abstracts
Example: a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , mode...


In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files("ccdv/pubmed-summarization", repo_type="dataset")
print(files[:5])

['.gitattributes', 'README.md', 'document/test-00000-of-00001.parquet', 'document/train-00000-of-00005.parquet', 'document/train-00001-of-00005.parquet']


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "parquet",
    data_files="hf://datasets/ccdv/pubmed-summarization/**/*.parquet",
    split="train[:5000]"
)
print(ds[0]["article"][:200])

Generating train split: 0 examples [00:00, ? examples/s]

anxiety affects quality of life in those living with parkinson 's disease ( pd ) more so than overall cognitive status , motor deficits , apathy , and depression [ 13 ] . although anxiety and depressi


In [ ]:
from datasets import load_dataset

# Example: load the MedQA dataset (board exams QA)
ds = load_dataset("bigbio/med_qa", split="train")
print(ds)            # Info about the dataset
print(ds[0])         # A single example

# Another example: load MEDIQA QA
ds2 = load_dataset("bigbio/mediqa_qa", split="train")
print(ds2[0])

The repository for bigbio/med_qa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/med_qa.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split: 0 examples [00:00, ? examples/s]

/root/.cache/huggingface/datasets/downloads/extracted/a6d27db281f920eeb16bb7391cacd81b4bb1f99493219f1ad0ae50fe700c7009/data_clean/questions/US/train.jsonl


Generating test split: 0 examples [00:00, ? examples/s]

/root/.cache/huggingface/datasets/downloads/extracted/a6d27db281f920eeb16bb7391cacd81b4bb1f99493219f1ad0ae50fe700c7009/data_clean/questions/US/test.jsonl


...

Generating train_live_qa_med split: 0 examples [00:00, ? examples/s]

Generating train_alexa split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

ValueError: Unknown split "train". Should be one of ['train_live_qa_med', 'train_alexa', 'validation', 'test'].


### Example 2: Code Dataset (Python)

In [ ]:
from datasets import load_dataset

# CodeParrot's curated Python dataset
dataset = load_dataset(
    "codeparrot/codeparrot-clean",
    split="train",
    streaming=True
)

code_texts = []
for i, item in enumerate(dataset):
    if i >= 10000:
        break
    code_texts.append(item['content'])

print(f"Collected {len(code_texts)} Python files")

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Collected 10000 Python files


For below codes, if you see NameError: name 'DataPipeline' is not defined, please go up to re-run Part4 Textcleaner and Part-5 DataPipline classes, or you can simply escape

In [ ]:
# Process
pipeline = DataPipeline()
data = pipeline.process(code_texts, min_length=200, max_length=1024)
pipeline.save(data, "./python_dataset")

Starting with 10,000 documents
Cleaning...
Filtering documents shorter than 200 characters...
  Remaining: 10,000
Deduplicating...
  Removed 18 duplicates
Tokenizing...
Total tokens: 8,072,607
Saved to ./python_dataset/



### Example 3: Custom Domain (Your Company Docs)

In [ ]:
# Scrape your documentation
import requests
from bs4 import BeautifulSoup

def scrape_documentation(base_url: str, max_pages: int = 100) -> List[str]:
    """Scrape documentation pages"""
    texts = []
    visited = set()
    to_visit = [base_url]

    while to_visit and len(texts) < max_pages:
        url = to_visit.pop(0)
        if url in visited:
            continue

        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract text
            for element in soup(['nav', 'header', 'footer', 'script']):
                element.decompose()
            text = soup.get_text(separator='\n', strip=True)
            texts.append(text)
            visited.add(url)

            # Find more links
            for link in soup.find_all('a', href=True):
                full_url = requests.compat.urljoin(base_url, link['href'])
                if full_url.startswith(base_url) and full_url not in visited:
                    to_visit.append(full_url)

        except Exception as e:
            print(f"Error scraping {url}: {e}")
            continue

    return texts

# Example usage
docs = scrape_documentation("https://example.com", max_pages=50)

# Process
pipeline = DataPipeline()
data = pipeline.process(docs, min_length=100, max_length=512)
pipeline.save(data, "./company_docs_dataset")

Starting with 1 documents
Cleaning...
Filtering documents shorter than 100 characters...
  Remaining: 1
Deduplicating...
  Removed 0 duplicates
Tokenizing...
Total tokens: 35
Saved to ./company_docs_dataset/


## Part 7: Quality Checks

Before training, verify your data quality.

In [ ]:
def analyze_dataset(data: Dict, tokenizer):
    """Analyze processed dataset quality"""
    train_tokens = data['train'].tolist()

    # Basic stats
    print("Dataset Statistics")
    print("=" * 50)
    print(f"Total training tokens: {len(train_tokens):,}")
    print(f"Vocabulary size: {data['vocab_size']:,}")
    print(f"Documents: {data['num_documents']:,}")

    # Token distribution
    unique_tokens = len(set(train_tokens))
    print(f"\nUnique tokens used: {unique_tokens:,} ({unique_tokens/data['vocab_size']*100:.1f}% of vocab)")

    # Sample sequences
    print("\nSample sequences:")
    for i in range(0, min(3000, len(train_tokens)), 1000):
        sequence = train_tokens[i:i+50]
        text = tokenizer.decode(sequence)
        print(f"\n[Position {i:,}]")
        print(text[:200])

    # Check for issues
    print("\nQuality Checks:")

    # 1. Repetition check
    def has_repetition(tokens, window=10):
        for i in range(len(tokens) - window * 2):
            if tokens[i:i+window] == tokens[i+window:i+window*2]:
                return True
        return False

    sample = train_tokens[:10000]
    if has_repetition(sample):
        print("⚠️  Warning: Detected repetitive patterns")
    else:
        print("✓ No obvious repetitive patterns")

    # 2. Diversity check
    diversity = unique_tokens / len(train_tokens[:10000])
    print(f"✓ Token diversity: {diversity:.3f} (higher is better)")

    # 3. Length check
    avg_tokens_per_doc = len(train_tokens) / data['num_documents']
    print(f"✓ Average tokens per document: {avg_tokens_per_doc:.1f}")

# Run analysis
analyze_dataset(data, pipeline.tokenizer)

Dataset Statistics
Total training tokens: 31
Vocabulary size: 50,257
Documents: 1

Unique tokens used: 24 (0.0% of vocab)

Sample sequences:

[Position 0]
Example Domain Example Domain This domain is for use in illustrative examples in documents. You may use this domain in literature without prior coordination or asking for permission.

Quality Checks:
✓ No obvious repetitive patterns
✓ Token diversity: 0.774 (higher is better)
✓ Average tokens per document: 31.0



## Part 8: Common Issues and Solutions

### Issue 1: Dataset Too Large for Memory

**Problem:** Can't load 50GB Wikipedia into RAM.

**Solution:** Use streaming or chunking.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# Create output directory
os.makedirs("./chunks", exist_ok=True)

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# ✅ Updated: Use current Wikipedia dataset
dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",  # Updated version
    split="train",
    streaming=True
)

def process_streaming(dataset, chunk_size=1000, total_chunks=10):
    all_tokens = []
    chunk_num = 0

    for i, item in enumerate(dataset):
        if i >= chunk_size * total_chunks:
            break

        # Extract text
        text = item['text']

        # Tokenize
        tokens = tokenizer.encode(text, max_length=512, truncation=True)
        all_tokens.extend(tokens)

        # Save periodically
        if (i + 1) % chunk_size == 0:
            chunk_num = (i + 1) // chunk_size
            torch.save(
                torch.tensor(all_tokens),
                f"./chunks/chunk_{chunk_num}.pt"
            )
            print(f"Saved chunk {chunk_num} with {len(all_tokens):,} tokens")
            all_tokens = []  # Clear memory

    # ✅ Save final chunk if there are remaining tokens
    if all_tokens:
        chunk_num += 1
        torch.save(
            torch.tensor(all_tokens),
            f"./chunks/chunk_{chunk_num}.pt"
        )
        print(f"Saved final chunk {chunk_num} with {len(all_tokens):,} tokens")

    return True

# Process the data
process_streaming(dataset)
print("✅ Processing complete!")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Saved chunk 1 with 489,735 tokens
Saved chunk 2 with 489,533 tokens
Saved chunk 3 with 453,004 tokens
Saved chunk 4 with 392,621 tokens
Saved chunk 5 with 405,888 tokens
Saved chunk 6 with 409,877 tokens
Saved chunk 7 with 401,924 tokens
Saved chunk 8 with 398,183 tokens
Saved chunk 9 with 399,199 tokens
Saved chunk 10 with 362,610 tokens
✅ Processing complete!



### Issue 2: Unbalanced Dataset

**Problem:** 90% of data is one category, 10% another.

**Solution:** Oversample minority or undersample majority.

In [ ]:
def balance_dataset(texts: List[str], labels: List[int]) -> tuple:
    """Balance dataset by oversampling minority class"""
    from collections import Counter
    import random

    # Count labels
    counts = Counter(labels)
    max_count = max(counts.values())

    # Oversample
    balanced_texts = []
    balanced_labels = []

    for label in set(labels):
        label_texts = [t for t, l in zip(texts, labels) if l == label]

        # Repeat until reaching max_count
        while len(label_texts) < max_count:
            label_texts.extend(random.sample(label_texts, min(len(label_texts), max_count - len(label_texts))))

        balanced_texts.extend(label_texts[:max_count])
        balanced_labels.extend([label] * max_count)

    return balanced_texts, balanced_labels

### Issue 3: Non-English or Mixed Language

**Problem:** Training on multiple languages or non-Latin scripts.

**Solution:** Use multilingual tokenizer or train your own.

In [ ]:
# Use multilingual BERT tokenizer
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

# Test with multiple languages
texts = [
    "Hello world",  # English
    "Bonjour le monde",  # French
    "こんにちは世界",  # Japanese
    "مرحبا بالعالم",  # Arabic
]

for text in texts:
    tokens = tokenizer.encode(text)
    print(f"{text} -> {len(tokens)} tokens")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

Hello world -> 4 tokens
Bonjour le monde -> 7 tokens
こんにちは世界 -> 9 tokens
مرحبا بالعالم -> 9 tokens
